In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_keyword = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/seo_keyword_performance/seo_keyword_performance.csv")
)

print("Rows:", df_keyword.count())
print("Columns:", len(df_keyword.columns))

display(df_keyword.limit(10))

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 4, Finished, Available, Finished, False)

Rows: 30000
Columns: 11


SynapseWidget(Synapse.DataFrame, 996244f8-3ee2-40c0-8040-1532c3313f45)

In [3]:
print("Country values:")
df_keyword.groupBy("country_code").count().orderBy("country_code").show()

print("Device values:")
df_keyword.groupBy("device_type").count().orderBy("device_type").show()

print("Ranking position statistics:")
df_keyword.select(
    F.min("ranking_position").alias("min_position"),
    F.max("ranking_position").alias("max_position")
).show()

print("Null ranking positions:")
print(
    df_keyword.filter(
        F.col("ranking_position").isNull()
    ).count()
)

print("Zero impressions:")
print(
    df_keyword.filter(
        F.col("impressions") == 0
    ).count()
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 5, Finished, Available, Finished, False)

Country values:
+------------+-----+
|country_code|count|
+------------+-----+
|          AU| 3038|
|          BR| 2933|
|          CA| 2949|
|          DE| 3041|
|          FR| 3078|
|          IN| 3026|
|          JP| 3001|
|          MX| 3008|
|          UK| 2942|
|          US| 2984|
+------------+-----+

Device values:
+-----------+-----+
|device_type|count|
+-----------+-----+
|    Desktop|14387|
|     Mobile|14132|
|     Tablet| 1481|
+-----------+-----+

Ranking position statistics:
+------------+------------+
|min_position|max_position|
+------------+------------+
|         0.0|        50.0|
+------------+------------+

Null ranking positions:
1
Zero impressions:
1


In [4]:
df_keyword_clean = (
    df_keyword
    .withColumn(
        "country_code",
        F.when(
            F.upper(F.trim(F.col("country_code"))).isin(
                "USA", "UNITED STATES", "US"
            ),
            F.lit("US")
        )
        .otherwise(F.upper(F.trim(F.col("country_code"))))
    )
    .withColumn(
        "device_type",
        F.initcap(F.trim(F.col("device_type")))
    )
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 6, Finished, Available, Finished, False)

In [5]:
display(
    df_keyword_clean
    .groupBy("country_code", "device_type")
    .count()
    .orderBy("country_code", "device_type")
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38b2fcf1-636b-4f6d-8f0f-4424bc9ce62e)

In [6]:
df_keyword_clean = df_keyword_clean.filter(
    (F.col("search_volume") >= 0) &
    (F.col("clicks") >= 0) &
    (F.col("impressions") >= 0) &
    (F.col("ranking_position") > 0)
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 8, Finished, Available, Finished, False)

In [7]:
df_keyword_clean = df_keyword_clean.withColumn(
    "ctr",
    F.when(
        F.col("impressions") > 0,
        F.col("clicks") / F.col("impressions")
    ).otherwise(F.lit(0.0))
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 9, Finished, Available, Finished, False)

In [8]:
duplicate_keyword_ids = (
    df_keyword_clean
    .groupBy("keyword_performance_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate keyword performance IDs:")
print(duplicate_keyword_ids.count())

display(
    duplicate_keyword_ids
    .orderBy(F.col("count").desc())
    .limit(10)
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 10, Finished, Available, Finished, False)

Duplicate keyword performance IDs:
0


SynapseWidget(Synapse.DataFrame, cdada1aa-23cf-48eb-a913-c079591d4b5f)

In [9]:
window_keyword = Window.partitionBy(
    "keyword_performance_id"
).orderBy(
    F.col("updated_timestamp").desc()
)

df_keyword_clean = (
    df_keyword_clean
    .withColumn("_rn", F.row_number().over(window_keyword))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(
    "Rows after deduplication:",
    df_keyword_clean.count()
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 11, Finished, Available, Finished, False)

Rows after deduplication: 29998


In [10]:
df_keyword_clean = df_keyword_clean.withColumn(
    "ranking_bucket",
    F.when(
        F.col("ranking_position") <= 3,
        "Top 3"
    )
    .when(
        F.col("ranking_position") <= 10,
        "Page 1"
    )
    .when(
        F.col("ranking_position") <= 20,
        "11-20"
    )
    .when(
        F.col("ranking_position") <= 50,
        "21-50"
    )
    .when(
        F.col("ranking_position") <= 100,
        "51-100"
    )
    .otherwise("100+")
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 12, Finished, Available, Finished, False)

In [11]:
display(
    df_keyword_clean
    .groupBy("ranking_bucket")
    .count()
    .orderBy("ranking_bucket")
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9070b54-4a84-4e87-9f8b-d567c460a9e3)

In [12]:
print("Final row count:", df_keyword_clean.count())
print("Final column count:", len(df_keyword_clean.columns))

print("\nDuplicate IDs:")
print(
    df_keyword_clean
    .groupBy("keyword_performance_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nInvalid ranking positions:")
print(
    df_keyword_clean
    .filter(F.col("ranking_position") <= 0)
    .count()
)

print("\nNegative metrics:")
print(
    df_keyword_clean
    .filter(
        (F.col("search_volume") < 0) |
        (F.col("clicks") < 0) |
        (F.col("impressions") < 0)
    )
    .count()
)

print("\nNull ranking positions:")
print(
    df_keyword_clean
    .filter(F.col("ranking_position").isNull())
    .count()
)

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 14, Finished, Available, Finished, False)

Final row count: 29998
Final column count: 12

Duplicate IDs:
0

Invalid ranking positions:
0

Negative metrics:
0

Null ranking positions:
0


In [13]:
(
    df_keyword_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_keyword_performance")
)

print("Silver fact_keyword_performance table created successfully.")

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 15, Finished, Available, Finished, False)

Silver fact_keyword_performance table created successfully.


In [14]:
df_silver_keyword = spark.read.table(
    "fact_keyword_performance"
)

print(
    "Silver keyword table rows:",
    df_silver_keyword.count()
)

print(
    "Silver keyword table columns:",
    len(df_silver_keyword.columns)
)

display(df_silver_keyword.limit(10))

StatementMeta(, a5d2fa01-2cbc-47d7-9aac-7d076dedf9dc, 16, Finished, Available, Finished, False)

Silver keyword table rows: 29998
Silver keyword table columns: 12


SynapseWidget(Synapse.DataFrame, a4af1785-07e9-43ad-a88b-e4c5fc5a0814)